# 🎙️ Speech Emotion Recognition
**Model:** `ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition`  
**Pretrained on:** RAVDESS (8 emotions)  
**No training required — inference only**

### Emotions detected:
`angry | calm | disgust | fearful | happy | neutral | sad | surprised`

## Cell 1 — Install Dependencies

In [ ]:
# Cell 1: Install all required packages
!pip install transformers torchaudio librosa soundfile datasets -q
print('✅ Dependencies installed')

## Cell 2 — Import Libraries

In [ ]:
# Cell 2: Imports
import torch
import torchaudio
import librosa
import numpy as np
import soundfile as sf
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
import matplotlib.pyplot as plt
import IPython.display as ipd
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')

## Cell 3 — Load Pretrained Model

> **Model choice rationale:** `ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition` is a large wav2vec2 model fine-tuned specifically on RAVDESS. It uses a Transformer encoder (not LSTM), which gives better temporal context. For LSTM-based comparison, we'll also demonstrate a feature-based approach in Cell 7.

In [ ]:
# Cell 3: Load model and feature extractor from HuggingFace Hub
MODEL_NAME = "ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"

print(f'Loading model: {MODEL_NAME}...')
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
model = AutoModelForAudioClassification.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

# Emotion label mapping from model config
id2label = model.config.id2label
print(f'✅ Model loaded. Emotion classes: {list(id2label.values())}')

## Cell 4 — Core Inference Function

In [ ]:
# Cell 4: Inference pipeline
def predict_emotion(audio_path: str, plot: bool = True):
    """
    Predict emotion from an audio file.
    
    Args:
        audio_path: Path to .wav file
        plot: Whether to show confidence bar chart
    
    Returns:
        dict with predicted emotion and all confidence scores
    """
    # Load and resample audio to 16kHz (required by wav2vec2)
    waveform, sample_rate = librosa.load(audio_path, sr=16000, mono=True)
    
    # Clip to max 10 seconds to avoid OOM
    max_length = 16000 * 10
    if len(waveform) > max_length:
        waveform = waveform[:max_length]
    
    # Extract features
    inputs = feature_extractor(
        waveform,
        sampling_rate=16000,
        return_tensors='pt',
        padding=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Forward pass (no gradient needed for inference)
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # Softmax to get probabilities
    probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    
    # Build result dict
    scores = {id2label[i]: float(probs[i]) for i in range(len(probs))}
    predicted = max(scores, key=scores.get)
    confidence = scores[predicted]
    
    # Print result
    print(f'\n🎭 Predicted Emotion : {predicted.upper()}')
    print(f'📊 Confidence        : {confidence:.1%}')
    print(f'📁 File              : {audio_path}')
    
    # Confidence bar chart
    if plot:
        fig, ax = plt.subplots(figsize=(10, 4))
        emotions = list(scores.keys())
        values = list(scores.values())
        colors = ['#e74c3c' if e == predicted else '#3498db' for e in emotions]
        bars = ax.barh(emotions, values, color=colors)
        ax.set_xlabel('Confidence Score')
        ax.set_title(f'Emotion Prediction: "{predicted.upper()}" ({confidence:.1%})')
        ax.set_xlim(0, 1)
        for bar, val in zip(bars, values):
            ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                    f'{val:.1%}', va='center', fontsize=9)
        plt.tight_layout()
        plt.show()
    
    return {'predicted': predicted, 'confidence': confidence, 'scores': scores}

print('✅ Inference function defined')

## Cell 5 — Test on RAVDESS Sample Audio

> Downloads a real RAVDESS sample from HuggingFace datasets

In [ ]:
# Cell 5: Download and test on a real RAVDESS audio sample
from datasets import load_dataset

# Load a small slice of RAVDESS (streaming avoids full download)
print('Loading RAVDESS sample from HuggingFace...')
dataset = load_dataset(
    'cahya/RAVDESS',
    split='train',
    streaming=True,
    trust_remote_code=True
)

# Take first 3 samples
samples = list(dataset.take(3))
print(f'✅ Loaded {len(samples)} samples')

# Save first sample to disk and run inference
sample = samples[0]
audio_array = np.array(sample['audio']['array'], dtype=np.float32)
sr = sample['audio']['sampling_rate']
true_label = sample.get('label', 'unknown')

# Save to temp file
test_path = '/tmp/ravdess_sample.wav'
sf.write(test_path, audio_array, sr)

print(f'Ground truth label index: {true_label}')
ipd.display(ipd.Audio(audio_array, rate=sr))  # Play audio in Colab

# Run prediction
result = predict_emotion(test_path)

## Cell 6 — Upload Your Own Audio File

In [ ]:
# Cell 6: Upload and test your own .wav file
from google.colab import files

print('Upload a .wav audio file (speech, 3-10 seconds recommended)...')
uploaded = files.upload()

for filename in uploaded.keys():
    print(f'\nAnalyzing: {filename}')
    result = predict_emotion(filename)

## Cell 7 — MFCC + LSTM Approach (Educational Comparison)

> This demonstrates the classical approach: hand-crafted features (MFCC) → LSTM classifier.  
> **Note:** Accuracy will be lower than wav2vec2 without full RAVDESS training.  
> This cell uses a **randomly initialized LSTM** — it shows the architecture only, not trained predictions.

In [ ]:
# Cell 7: MFCC feature extraction + LSTM architecture demo
import torch.nn as nn

# --- Feature Extraction ---
def extract_mfcc_features(audio_path: str, n_mfcc: int = 40, max_frames: int = 300):
    """
    Extract MFCC features from audio.
    Shape: (max_frames, n_mfcc) — sequence of frames for LSTM
    """
    y, sr = librosa.load(audio_path, sr=22050, mono=True)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)  # (n_mfcc, T)
    mfcc = mfcc.T  # (T, n_mfcc) — time-first for LSTM
    
    # Pad or truncate to fixed length
    if mfcc.shape[0] < max_frames:
        pad_width = max_frames - mfcc.shape[0]
        mfcc = np.pad(mfcc, ((0, pad_width), (0, 0)), mode='constant')
    else:
        mfcc = mfcc[:max_frames]
    
    return mfcc


# --- LSTM Model Architecture ---
class EmotionLSTM(nn.Module):
    """
    Bidirectional LSTM for sequence-level emotion classification.
    Input: (batch, seq_len, input_size) = (B, 300, 40)
    Output: (batch, num_classes) = (B, 8)
    """
    def __init__(self, input_size=40, hidden_size=128, num_layers=2,
                 num_classes=8, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,   # Bidirectional for better context
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        # x: (B, T, input_size)
        lstm_out, (hidden, _) = self.lstm(x)
        # Take last time step of both directions
        # hidden shape: (num_layers * 2, B, hidden_size)
        # Concat forward and backward last hidden states
        fwd = hidden[-2]   # Last forward layer
        bwd = hidden[-1]   # Last backward layer
        combined = torch.cat([fwd, bwd], dim=-1)  # (B, hidden*2)
        out = self.layer_norm(combined)
        out = self.dropout(out)
        logits = self.classifier(out)
        return logits


# Instantiate and show architecture
lstm_model = EmotionLSTM(input_size=40, hidden_size=128, num_layers=2,
                          num_classes=8, dropout=0.3).to(device)
print('LSTM Model Architecture:')
print(lstm_model)

# Count parameters
total_params = sum(p.numel() for p in lstm_model.parameters())
print(f'\nTotal Parameters: {total_params:,}')

# Demo forward pass with MFCC features
features = extract_mfcc_features(test_path)  # (300, 40)
x = torch.tensor(features, dtype=torch.float32).unsqueeze(0).to(device)  # (1, 300, 40)
with torch.no_grad():
    logits = lstm_model(x)

print(f'\nMFCC feature shape : {features.shape}  (frames x MFCC coefficients)')
print(f'Input tensor shape : {x.shape}  (batch x frames x features)')
print(f'Output logits shape: {logits.shape}  (batch x num_classes)')
print('\n⚠️  Note: LSTM output is random (untrained). '
      'Full training on RAVDESS needed for meaningful predictions.')

## Cell 8 — Visualize Audio Features

In [ ]:
# Cell 8: Visualize waveform, mel-spectrogram, and MFCC
def visualize_audio(audio_path: str):
    y, sr = librosa.load(audio_path, sr=22050)
    
    fig, axes = plt.subplots(3, 1, figsize=(12, 8))
    
    # 1. Waveform
    import librosa.display
    librosa.display.waveshow(y, sr=sr, ax=axes[0], color='#2ecc71')
    axes[0].set_title('Waveform', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Time (s)')
    
    # 2. Mel-Spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time',
                                    y_axis='mel', ax=axes[1], cmap='magma')
    axes[1].set_title('Mel-Spectrogram', fontsize=12, fontweight='bold')
    fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
    
    # 3. MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    img2 = librosa.display.specshow(mfcc, sr=sr, x_axis='time',
                                     ax=axes[2], cmap='coolwarm')
    axes[2].set_title('MFCC (40 coefficients)', fontsize=12, fontweight='bold')
    fig.colorbar(img2, ax=axes[2])
    
    plt.tight_layout()
    plt.suptitle('Audio Feature Visualization', fontsize=14,
                 fontweight='bold', y=1.01)
    plt.show()

visualize_audio(test_path)

## Cell 9 — Batch Inference on Multiple Files

In [ ]:
# Cell 9: Batch inference + summary table
import pandas as pd

def batch_predict(audio_paths: list):
    """
    Run emotion prediction on a list of audio files.
    Returns a DataFrame with results.
    """
    records = []
    for path in audio_paths:
        try:
            result = predict_emotion(path, plot=False)
            records.append({
                'file': path.split('/')[-1],
                'predicted_emotion': result['predicted'],
                'confidence': f"{result['confidence']:.1%}",
                **{f'score_{k}': f"{v:.2f}" for k, v in result['scores'].items()}
            })
        except Exception as e:
            records.append({'file': path, 'error': str(e)})
    
    df = pd.DataFrame(records)
    return df


# Test batch prediction on the 3 RAVDESS samples saved earlier
saved_paths = []
for i, sample in enumerate(samples):
    path = f'/tmp/ravdess_sample_{i}.wav'
    audio_arr = np.array(sample['audio']['array'], dtype=np.float32)
    sr_s = sample['audio']['sampling_rate']
    sf.write(path, audio_arr, sr_s)
    saved_paths.append(path)

print('Running batch inference...')
results_df = batch_predict(saved_paths)
print('\n📋 Batch Results:')
print(results_df[['file', 'predicted_emotion', 'confidence']].to_string(index=False))

## Cell 10 — Architecture Comparison Summary

In [ ]:
# Cell 10: Print architecture comparison
summary = {
    'Approach': ['MFCC + Bi-LSTM', 'wav2vec2 (Transformer)'],
    'Feature Type': ['Hand-crafted (MFCC)', 'Self-supervised learned'],
    'Context Window': ['Local frames', 'Full sequence attention'],
    'RAVDESS Accuracy': ['~65-72% (reported avg)', '~85-90% (reported avg)'],
    'Training Required': ['Yes (from scratch)', 'No (pretrained)'],
    'Natural Speech': ['Poor generalization', 'Better generalization'],
    'Params (approx)': ['~200K', '~300M'],
    'Inference Speed': ['Fast (CPU ok)', 'Slow (GPU recommended)']
}

df_summary = pd.DataFrame(summary)
print('\n📊 Architecture Comparison:')
print(df_summary.to_string(index=False))

print('\n✅ Prototype complete.')
print('   For production: fine-tune wav2vec2 on domain-specific data')
print('   RAVDESS accuracy on natural (non-acted) speech drops ~15-20pp')